# Homework 2 — Data Preprocessing

> **Study/reference notebook:** The assignment asks students to write their own code. Use this notebook to study the methods, check the required return formats, and adapt the solution yourself in the Coursera notebook. Do not include your name in a project title.

This assignment practices Chapter 3 topics: data transformation through normalization, Pearson correlation analysis, and data visualization for exploring attributes.

## Assignment rules and imports

- Do not install additional packages beyond those included in the Coursera data science Docker image.
- Preserve the function signatures and supplied data locations.
- Use relative paths to access the datasets.
- The Coursera-provided CSV and pickle test files are not included in this notebook.

The functions below follow the assignment's positional `attr` argument: it selects a column by zero-based index. For the financial data the assignment describes the fields as Open, High, Low, Close, and Volume; inspect the supplied CSV header in Coursera if its exact ordering or labels differ.

In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import math
import matplotlib.pyplot as plt
import matplotlib

# Plot appearance specified by the assignment.
matplotlib.rc('figure', figsize=(14, 7))
matplotlib.rc('font', size=14)
matplotlib.rc('axes.spines', top=False, right=False)
matplotlib.rc('axes', grid=False, facecolor='white')

## Problem 1 — normalization and correlation

### Normalization

Min–max normalization rescales values to `[0, 1]` using `(x - min) / (max - min)`. Z-score normalization centers values at the mean and divides by the standard deviation. The function returns a dictionary mapping each original non-missing value to its normalized value, as required. For a constant-valued column, it returns zero for every value because the usual denominator is zero.

### Correlation

Pearson correlation measures the direction and strength of a linear relationship. As emphasized in Chapter 3, correlation alone does not establish causality. Rows with a missing value in either selected attribute are excluded from the paired calculation.

In [ ]:
def normalization(fname, attr, normType):
    '''
    Return a dictionary mapping original values in column `attr` to
    min-max or z-score normalized values.
    '''
    df = pd.read_csv(fname)
    original = df.iloc[:, attr]

    # Convert values to numbers while allowing common formatted CSV values
    # such as currency strings or comma-separated quantities.
    numeric = pd.to_numeric(
        original.astype(str).str.replace(',', '', regex=False).str.replace('$', '', regex=False),
        errors='coerce'
    )
    valid = numeric.notna()
    values = numeric[valid]

    if normType == 'min_max':
        span = values.max() - values.min()
        normalized = (values - values.min()) / span if span != 0 else values * 0
    elif normType == 'z_score':
        std = values.std()  # pandas sample standard deviation (ddof=1)
        normalized = (values - values.mean()) / std if pd.notna(std) and std != 0 else values * 0
    else:
        raise ValueError("normType must be 'min_max' or 'z_score'")

    result = {}
    for original_value, normalized_value in zip(original[valid], normalized):
        result[original_value] = float(normalized_value)
    return result

In [ ]:
def correlation(fname1, attr1, fname2, attr2):
    '''Return the Pearson correlation of two selected numeric columns.'''
    df1 = pd.read_csv(fname1)
    df2 = pd.read_csv(fname2)

    x = pd.to_numeric(df1.iloc[:, attr1], errors='coerce')
    y = pd.to_numeric(df2.iloc[:, attr2], errors='coerce')

    # Correlation is calculated for paired rows, so both series must be
    # trimmed to positions where both values are present.
    paired = pd.concat([x.reset_index(drop=True), y.reset_index(drop=True)], axis=1).dropna()
    if len(paired) < 2:
        return float('nan')
    correlation_coefficient = paired.iloc[:, 0].corr(paired.iloc[:, 1])
    return float(correlation_coefficient)

### Problem 1 data checks

Use the CSV files and unit-test cells supplied in Coursera. In the prompt, the visible checks use `data/normalization_test_data.csv` for both normalization modes and `data/correlation_test_data.csv` for correlation. Other sample files are also listed in the prompt. Do not create or move substitute files in the graded environment.

In [ ]:
# Example calls when the Coursera data files are available.
# normalized = normalization('./data/normalization_test_data.csv', 0, 'min_max')
# corr = correlation('./data/correlation_test_data.csv', 0,
#                    './data/correlation_test_data.csv', 0)
# normalized, corr

## Problem 2 — plots for historical quotes

The assignment asks for four functions using `./data/HistoricalQuotes.csv`. The first three functions return values and labels in the exact forms described in their docstrings. `amzn_new_plot()` is open-ended: it should display a plot of your choice using the same dataset.

In [ ]:
def temporal_graph():
    '''Return dates and (High, Low) lists for a temporal graph.'''
    df = pd.read_csv('./data/HistoricalQuotes.csv')

    x_data = list(df['Date'])
    y_data = (list(df['High']), list(df['Low']))
    xlabel = 'Date'
    ylabel = 'Price'
    title = 'High and Low Prices Over Time'

    return x_data, y_data, xlabel, ylabel, title

In [ ]:
def boxplot():
    '''Return Open and Close lists and labels for a boxplot.'''
    df = pd.read_csv('./data/HistoricalQuotes.csv')

    x_data = (list(df['Open']), list(df['Close']))
    xlabel = 'Attribute'
    ylabel = 'Price'
    title = 'Distribution of Open and Close Prices'

    return x_data, xlabel, ylabel, title

In [ ]:
def histogram():
    '''Return Volume values and axis labels for a histogram.'''
    df = pd.read_csv('./data/HistoricalQuotes.csv')

    data = list(df['Volume'])
    x_label = 'Volume'
    y_label = 'Frequency'

    return data, x_label, y_label

### Custom plot

This example uses a line plot of closing price by date. It also cleans common formatting characters from price strings if the CSV stores them that way.

In [ ]:
def amzn_new_plot():
    '''Display a line plot of the closing price over time.'''
    df = pd.read_csv('./data/HistoricalQuotes.csv')
    close = pd.to_numeric(
        df['Close'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
        errors='coerce'
    )
    dates = pd.to_datetime(df['Date'], errors='coerce')

    plt.figure()
    plt.plot(dates, close)
    plt.title('AMZN Closing Price Over Time')
    plt.xlabel('Date')
    plt.ylabel('Closing Price')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Display the requested plots

These calls use the return formats required in the prompt and render the corresponding figures. They require the Coursera `HistoricalQuotes.csv` file.

In [ ]:
x_data, y_data, xlabel, ylabel, title = temporal_graph()
plt.figure()
plt.plot(x_data, y_data[0], label='High')
plt.plot(x_data, y_data[1], label='Low')
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.title(title)
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
x_data, xlabel, ylabel, title = boxplot()
plt.figure()
plt.boxplot(x_data, labels=['Open', 'Close'])
plt.xlabel(xlabel)
plt.ylabel(ylabel)
plt.title(title)
plt.show()

In [ ]:
data, x_label, y_label = histogram()
plt.figure()
plt.hist(data, bins=20)
plt.xlabel(x_label)
plt.ylabel(y_label)
plt.title('Distribution of Volume')
plt.show()

In [ ]:
amzn_new_plot()

## Final checklist

- Check the actual CSV headers and types in the Coursera environment.
- Keep all given signatures and relative data paths unchanged.
- Run the visible tests supplied with the assignment; hidden tests cannot be run from this local notebook.
- Pearson correlation describes linear association, not cause and effect.
- Ensure each plot function returns exactly the tuple/list structure specified by its assignment docstring.